# Zeek Log Cleaning (zat-based)

This notebook reads common Zeek log types using **zat**, applies light normalization + stable pseudonymization (reusing `mappings/`), and writes cleaned outputs to `data/cleaned_zeek/`.

Supported log types:
- `conn.log`
- `dns.log`
- `http.log`
- `ssl.log`
- `files.log`
- `ssh.log`


In [12]:
# --- 0) Install & import dependencies (zat) ---
import sys, subprocess
import os, json, hashlib, re
from pathlib import Path
import pandas as pd
from zat.log_to_dataframe import LogToDataFrame
from collections import defaultdict

pd.set_option("display.max_columns", 200)


In [13]:
# --- 1) Paths & log list ---
PROJECT_ROOT = Path.cwd().parent.resolve()

DATA_DIR = PROJECT_ROOT / "data"         # put *.log here
OUT_DIR = PROJECT_ROOT / "sanidata"
MAP_DIR = PROJECT_ROOT / "mappings"    # reuse existing mapping dir

OUT_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

# --- 1) Zeek log "types" we care about (filename-based) ---
ZEEK_FILES = {
    "conn":  "conn.log",
    "dns":   "dns.log",
    "http":  "http.log",
    "ssl":   "ssl.log",
    "files": "files.log",
    "ssh":   "ssh.log",
}

def extract_scenario(path: Path) -> str:
    """
    Extract scenario id like 'sc1' from any part of the path.
    Falls back to 'unknown' if not found.
    """
    for part in path.parts:
        if part.startswith("sc") and part[2:].isdigit():
            return part
    return "unknown"

def collect_inputs(base: Path, files: dict) -> dict:
    """
    Return:
      inputs[scenario][log_type] -> list[Path]  (1 or many)
    """
    inputs = defaultdict(lambda: defaultdict(list))
    for log_type, fname in files.items():
        for p in base.rglob(fname):
            sc = extract_scenario(p)
            inputs[sc][log_type].append(p)
    return inputs

INPUTS = collect_inputs(DATA_DIR, ZEEK_FILES)

# Quick sanity print: how many matches per scenario/log_type
for sc in sorted(INPUTS):
    counts = {t: len(ps) for t, ps in INPUTS[sc].items()}
    print(f"{sc}: {counts}")


print("Input dir:", DATA_DIR)
print("Output dir:", OUT_DIR)
print("Mappings dir:", MAP_DIR)


sc3: {'conn': 1, 'dns': 1, 'http': 1, 'ssl': 1, 'files': 1, 'ssh': 1}
sc4: {'conn': 1, 'dns': 1, 'http': 1, 'ssl': 1, 'files': 1, 'ssh': 1}
sc7: {'conn': 1, 'dns': 1, 'http': 1, 'ssl': 1, 'files': 1, 'ssh': 1}
Input dir: /Users/zhuoran/Projects/SSCMDataset/data
Output dir: /Users/zhuoran/Projects/SSCMDataset/sanidata
Mappings dir: /Users/zhuoran/Projects/SSCMDataset/mappings


In [16]:
# --- 2) Stable SALT (reuse same pattern) ---
SALT_PATH = MAP_DIR / "salt.txt"
if SALT_PATH.exists():
    SALT = SALT_PATH.read_text().strip()
else:
    SALT = hashlib.sha256(os.urandom(32)).hexdigest()
    SALT_PATH.write_text(SALT)

# --- 3) ZAT reader ---
log_reader = LogToDataFrame()

In [17]:


def _load_map(name: str) -> dict:
    p = MAP_DIR / f"{name}.json"
    if p.exists():
        return json.loads(p.read_text())
    return {}

def _save_map(name: str, m: dict):
    p = MAP_DIR / f"{name}.json"
    p.write_text(json.dumps(m, ensure_ascii=False, indent=2))

def stable_hash(text: str) -> str:
    return hashlib.sha256((SALT + str(text)).encode("utf-8")).hexdigest()

def map_token(value, prefix: str, map_name: str, digits: int = 6):
    """Map a raw value to a stable pseudonym like ip_012345, user_000123, etc.
    Uses mappings/<map_name>.json for persistence across runs.
    """
    if value is None:
        return None
    if isinstance(value, float) and pd.isna(value):
        return None
    s = str(value).strip()
    if s == "" or s == "-" or s.lower() in {"(empty)", "null", "none", "nan"}:
        return None

    m = _load_map(map_name)
    if s in m:
        return m[s]

    base = int(stable_hash(s)[:8], 16) % (10**digits)
    candidate = f"{prefix}_{base:0{digits}d}"

    used = set(m.values())
    i = 0
    while candidate in used:
        i += 1
        candidate = f"{prefix}_{(base+i)%(10**digits):0{digits}d}"

    m[s] = candidate
    _save_map(map_name, m)
    return candidate

def normalize_scalar(x):
    """Normalize common Zeek placeholders."""
    if x is None:
        return None
    if isinstance(x, float) and pd.isna(x):
        return None
    if isinstance(x, str):
        s = x.strip()
        if s in {"-", ""}:
            return None
        if s.lower() in {"(empty)", "null", "none", "nan"}:
            return None
        return s
    return x


In [18]:

def read_zeek_log(path: Path) -> pd.DataFrame:
    df = log_reader.create_dataframe(str(path))
    # Standardize column names (zat already uses the #fields names)
    df.columns = [c.strip() for c in df.columns]
    return df


In [19]:
# --- 4) Cleaning & pseudonymization rules ---
IP_COL_PATTERNS = [
    re.compile(r"(^|\.)orig_h$"),
    re.compile(r"(^|\.)resp_h$"),
    re.compile(r"(^|\.)id\.orig_h$"),
    re.compile(r"(^|\.)id\.resp_h$"),
    re.compile(r"(^|\.)src$"),
    re.compile(r"(^|\.)dst$"),
    re.compile(r"(^|\.)client$"),
    re.compile(r"(^|\.)server$"),
]

USER_COL_CANDIDATES = {
    "user", "username", "auth_user", "client_user", "server_user", "uid_user"
}

# def pick_ip_cols(cols):
#     out = []
#     for c in cols:
#         for pat in IP_COL_PATTERNS:
#             if pat.search(c):
#                 out.append(c)
#                 break
#     # Common zeek field names in some pipelines
#     for c in ["id.orig_h", "id.resp_h", "orig_h", "resp_h"]:
#         if c in cols and c not in out:
#             out.append(c)
#     return out

def pick_user_cols(cols):
    out = [c for c in cols if c in USER_COL_CANDIDATES]
    # Some zeek logs have user-ish fields like "user_name" or "username"
    out += [c for c in cols if re.search(r"(?:^|\.)user(name)?$", c)]
    return sorted(set(out))

def clean_df(df: pd.DataFrame, log_type: str) -> pd.DataFrame:
    # Normalize placeholders
    for c in df.columns:
        if df[c].dtype == "object":
            df[c] = df[c].map(normalize_scalar)

    # # Pseudonymize IPs
    # ip_cols = pick_ip_cols(df.columns)
    # for c in ip_cols:
    #     df[c] = df[c].map(lambda v: map_token(v, "ip", "ip_map", digits=6))

    # Pseudonymize usernames (if present)
    user_cols = pick_user_cols(df.columns)
    for c in user_cols:
        df[c] = df[c].map(lambda v: map_token(v, "user", "user_map", digits=6))

    # Add a couple of helpful standard columns
    df.insert(0, "log_type", log_type.replace(".log",""))
    if "ts" in df.columns:
        # Zeek ts is epoch seconds (float). Convert to UTC timestamp for convenience.
        df["ts_utc"] = pd.to_datetime(df["ts"], unit="s", utc=True, errors="coerce")

    return df


In [22]:
# --- 5) Process all logs ---
results = {}

# --- 4) Example: iterate per scenario and read all matched logs ---
for sc, tables in INPUTS.items():
    sc_out = OUT_DIR / sc
    sc_out.mkdir(parents=True, exist_ok=True)

    for log_type, paths in tables.items():
        for idx, p in enumerate(sorted(paths)):
            df = log_reader.create_dataframe(str(p))

            df = clean_df(df.copy(), log_type)

            # write with collision-safe naming if multiple occurrences exist
            suffix = f"_{idx}" if len(paths) > 1 else ""
            out_csv = sc_out / f"{log_type}{suffix}_cleaned.csv"
            out_parq = sc_out / f"{log_type}{suffix}_cleaned.parquet"

            print("Wrote:", out_csv)
            print("Wrote:", out_parq)

            df.to_csv(out_csv, index=False)
            df.to_parquet(out_parq, engine="fastparquet", index=False)

            print(f"[{sc}] {log_type}{suffix}: {p} -> {out_csv.name}, {out_parq.name}")

print("\nDone. Cleaned log types:", list(results.keys()))


Wrote: /Users/zhuoran/Projects/SSCMDataset/sanidata/sc7/conn_cleaned.csv
Wrote: /Users/zhuoran/Projects/SSCMDataset/sanidata/sc7/conn_cleaned.parquet
[sc7] conn: /Users/zhuoran/Projects/SSCMDataset/data/sc7/conn.log -> conn_cleaned.csv, conn_cleaned.parquet
Wrote: /Users/zhuoran/Projects/SSCMDataset/sanidata/sc7/dns_cleaned.csv
Wrote: /Users/zhuoran/Projects/SSCMDataset/sanidata/sc7/dns_cleaned.parquet
[sc7] dns: /Users/zhuoran/Projects/SSCMDataset/data/sc7/dns.log -> dns_cleaned.csv, dns_cleaned.parquet
Wrote: /Users/zhuoran/Projects/SSCMDataset/sanidata/sc7/http_cleaned.csv
Wrote: /Users/zhuoran/Projects/SSCMDataset/sanidata/sc7/http_cleaned.parquet
[sc7] http: /Users/zhuoran/Projects/SSCMDataset/data/sc7/http.log -> http_cleaned.csv, http_cleaned.parquet
Wrote: /Users/zhuoran/Projects/SSCMDataset/sanidata/sc7/ssl_cleaned.csv
Wrote: /Users/zhuoran/Projects/SSCMDataset/sanidata/sc7/ssl_cleaned.parquet
[sc7] ssl: /Users/zhuoran/Projects/SSCMDataset/data/sc7/ssl.log -> ssl_cleaned.csv, 

In [24]:
# --- 6) Quick peek ---
for lt, df in results.items():
    print("\n===", lt, "===")
    display(df.head(3))
